# 06. Model Comparison and Transfer

A larger model is not automatically better. This module uses nested models and partial F tests to decide whether a block of predictors adds explanatory value beyond a reduced model.

By the end of this notebook, you should be able to:

- compare nested models with a partial F test;
- test a categorical predictor as a dummy-variable block;
- use parsimony and a transfer checklist to decide what model to carry into a new problem.


In [1]:
from lite_setup import ensure_packages
await ensure_packages()

from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import scipy.stats as st
import statsmodels.api as sm
import statsmodels.formula.api as smf

plt.rcParams["figure.figsize"] = (7, 4)
plt.rcParams["axes.grid"] = True


Running outside JupyterLite; assuming packages are already installed.


In [2]:
ads = pd.read_csv("data/advertising_sales.csv")
full = smf.ols("Sales ~ RadioSpend + PrintSpend", data=ads).fit()
reduced = smf.ols("Sales ~ RadioSpend", data=ads).fit()

The reduced model must be nested inside the full model. Here, the reduced model removes `PrintSpend` while keeping `RadioSpend`. The partial F test asks whether print advertising adds explanatory value after radio advertising is already in the model.

If the reduced model has $g$ predictors and the full model has $k$ predictors, the partial F statistic is

$$F_0=\frac{(SSE_R-SSE_F)/(k-g)}{SSE_F/(n-k-1)},$$

where $SSE_R$ is the reduced-model error sum of squares and $SSE_F$ is the full-model error sum of squares. Under the null hypothesis that the added predictors have zero coefficients, this follows an $F_{k-g,\,n-k-1}$ distribution.


In [3]:
sse_reduced = np.sum(reduced.resid**2)
sse_full = np.sum(full.resid**2)
q = reduced.df_resid - full.df_resid
partial_f = ((sse_reduced - sse_full) / q) / (sse_full / full.df_resid)
p_value = 1 - st.f.cdf(partial_f, q, full.df_resid)

print(f"Manual partial F = {partial_f:.4f}")
print(f"p-value = {p_value:.6g}")


Manual partial F = 158.2933
p-value = 4.86262e-10


In [4]:
print(f"Reduced adjusted R-squared: {reduced.rsquared_adj:.4f}")
print(f"Full adjusted R-squared:    {full.rsquared_adj:.4f}")


Reduced adjusted R-squared: 0.6029
Full adjusted R-squared:    0.9592


## Principle of Parsimony

A parsimonious model uses the smallest number of parameters that still answers the scientific or operational question well. When two models have essentially similar predictive performance and support the same decision, prefer the simpler model because it is easier to explain, less fragile, and usually less sensitive to noise.

Do not compare unrelated models only by saying one has a larger F statistic. The overall F statistic tests a model against the intercept-only baseline; it is not a universal model-ranking score.


## Partial F Test for a Categorical Predictor Block

A categorical predictor with more than two levels creates a block of dummy variables. The right question is often whether the entire block improves the model, not whether one dummy coefficient happens to be significant by itself.


In [5]:
stores = pd.read_csv("data/store_sales.csv")
reduced_location = smf.ols("SalesVolume ~ Households", data=stores).fit()
full_location = smf.ols('SalesVolume ~ Households + C(Location, Treatment(reference="Street"))', data=stores).fit()
sm.stats.anova_lm(reduced_location, full_location)


,df_resid,ssr,df_diff,ss_diff,F,Pr(>F)
0,22.0,4195.191977,0.0,NaN,NaN,NaN
1,20.0,48.055260,2.0,4147.136718,862.993301,3.889474e-20


This test compares the model with household count only to the model that also includes the location dummy-variable block. It is the same logic as the partial F test above, applied to all location indicators together.


## Transfer Checklist

When you use multiple regression on a new problem, use this sequence:

1. Define the response and each predictor, including units.
2. Plot the response against each quantitative predictor and inspect predictor-predictor relationships.
3. Fit a model that matches the scientific question.
4. Interpret coefficients conditionally.
5. Use t tests for individual coefficients and F tests for joint questions.
6. Use confidence intervals for mean response and prediction intervals for future observations.
7. Prefer simpler models unless the added terms answer a real question or improve predictive performance for a defensible reason.


## Mini Transfer Exercise

Use the mileage dataset below. Fit a simple regression of fuel mileage on additive percentage, then ask what would change if you had a second meaningful predictor. The transfer habit is the same: define the response, define the predictors, fit the model, inspect uncertainty, and explain the result in context.

In [6]:
mileage = pd.read_csv("data/mileage_additive.csv")
mileage_model = smf.ols("MPG ~ AdditivePercent", data=mileage).fit()
print(mileage_model.summary())

                            OLS Regression Results                            
Dep. Variable:                    MPG   R-squared:                       0.072
Model:                            OLS   Adj. R-squared:                 -0.044
Method:                 Least Squares   F-statistic:                    0.6177
Date:                Mon, 18 May 2026   Prob (F-statistic):              0.455
Time:                        00:58:41   Log-Likelihood:                -20.473
No. Observations:                  10   AIC:                             44.95
Df Residuals:                       8   BIC:                             45.55
Df Model:                           1                                         
Covariance Type:            nonrobust                                         
                      coef    std err          t      P>|t|      [0.025      0.975]
-----------------------------------------------------------------------------------
Intercept          28.8133      1.232     